# Занятие 7. Очистка данных: пропуски, дубликаты и типы данных

## Краткая теория по ноутбуку

На этом занятии мы начинаем работать с **грязными данными**. В реальной задаче таблица почти никогда не бывает идеальной:
- в некоторых ячейках есть пропуски;
- некоторые строки повторяются;
- числовые столбцы после загрузки могут иметь не тот тип, который нужен для расчётов.

Наша задача — научиться выполнять базовую очистку данных в `pandas`, чтобы после этого таблицу можно было анализировать, визуализировать и использовать в задачах ИИ.

## Что важно понять
- **Пропуск** — это пустое значение. В `pandas` оно часто отображается как `NaN`.
- **Дубликат** — это повторяющаяся строка.
- **Тип данных** показывает, как Python хранит значения: текст, целое число, число с плавающей точкой и т.д.
- После очистки данных удобно создавать новые вычисляемые столбцы, например `revenue`.

## Методы, которые будем использовать
- `pd.read_excel(...)` — загрузка Excel-файла.
- `df.isna().sum()` — подсчёт пропусков.
- `df.fillna(...)` — заполнение пропусков.
- `df.dropna(...)` — удаление строк с пропусками.
- `df.duplicated()` — поиск дубликатов.
- `df.drop_duplicates()` — удаление дубликатов.
- `df.astype(...)` — приведение столбца к нужному типу.

## Цель практики
1. Загрузить «грязную» таблицу из Excel.
2. Найти пропуски.
3. Заполнить пропуски и посмотреть пример удаления строк через `dropna`.
4. Найти и удалить дубликаты.
5. Привести типы данных и посчитать итоговую выручку.


## Практическая ячейка 1. Загружаем Excel и смотрим на структуру таблицы

### Что делаем
Открываем Excel-файл занятия, считываем лист `dirty_data` и смотрим:
- размер таблицы;
- названия столбцов;
- типы данных.

### Функции, методы и синтаксис
- `import pandas as pd` — подключаем библиотеку `pandas`.
- `Path("имя_файла.xlsx")` — создаём удобный путь к файлу.
- `pd.read_excel(file_path, sheet_name="dirty_data", header=1)` — читаем конкретный лист Excel в `DataFrame`.
- `df.shape` — показывает количество строк и столбцов.
- `df.dtypes` — показывает тип каждого столбца.
- `df` в конце ячейки — выводит саму таблицу.


In [30]:
import pandas as pd
from pathlib import Path
from google.colab import drive # Added import for drive

# Mount Google Drive
drive.mount('/content/drive') # Added drive mount

# Assuming the file is in the root of your Google Drive
# If your file is in a specific folder, e.g., 'Colab Notebooks', use:

file_path = Path("/content/drive/MyDrive/Базы/lesson_07_data_cleaning.xlsx") # Modified file path

df = pd.read_excel(file_path, sheet_name="dirty_data", header=1)

print("Размер таблицы:", df.shape)
print("\nНазвания столбцов:")
print(list(df.columns))
print("\nТипы данных до очистки:")
print(df.dtypes)

df

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Размер таблицы: (8, 8)

Названия столбцов:
['order_id', 'client', 'city', 'category', 'product', 'quantity', 'price', 'manager']

Типы данных до очистки:
order_id      int64
client       object
city         object
category     object
product      object
quantity    float64
price       float64
manager      object
dtype: object


,order_id,client,city,category,product,quantity,price,manager
0,101,ООО Альфа,Москва,Ноутбуки,Notebook Air,2.0,55000.0,Анна
1,102,ООО Бета,NaN,Мониторы,Монитор 24,1.0,18000.0,Борис
2,103,ИП Гамма,Уфа,Периферия,Мышь Pro,NaN,1500.0,Анна
3,104,ООО Дельта,Самара,Ноутбуки,Notebook Pro,1.0,NaN,Виктор
4,105,ИП Омега,Омск,Периферия,Клавиатура,4.0,3200.0,Анна
5,105,ИП Омега,Омск,Периферия,Клавиатура,4.0,3200.0,Анна
6,106,ООО Сигма,Казань,Мониторы,Монитор 27,3.0,21000.0,Елена
7,107,ООО Тета,Пермь,Периферия,Кабель HDMI,5.0,900.0,NaN


In [ ]:
from pathlib import Path

drive_path = Path("/content/drive/MyDrive/Базы")

# List all files and directories in MyDrive
# You can filter for specific file types by changing the pattern, e.g., '*.xlsx'
files_and_dirs = list(drive_path.glob('*'))

print("Files and directories in /content/drive/MyDrive:")
for item in files_and_dirs:
    print(item)

Files and directories in /content/drive/MyDrive:
/content/drive/MyDrive/Базы/Запись движения по улице Лондона
/content/drive/MyDrive/Базы/chest_xray.zip
/content/drive/MyDrive/Базы/reality
/content/drive/MyDrive/Базы/lesson_07_data_cleaning.xlsx


## Практическая ячейка 2. Ищем пропуски

### Что делаем
Смотрим, в каких столбцах есть пустые значения, и выводим строки, где есть хотя бы один пропуск.

### Функции, методы и синтаксис
- `df.isna()` — создаёт таблицу из `True/False`, где `True` означает пропуск.
- `df.isna().sum()` — считает количество пропусков по столбцам.
- `df.isna().any(axis=1)` — проверяет, есть ли в строке хотя бы один пропуск.
- `df[...]` — фильтрация строк по условию.


In [ ]:
missing_by_column = df.isna().sum()
rows_with_missing = df[df.isna().any(axis=1)]

print("Пропуски по столбцам:")
print(missing_by_column)

print("\nСтроки, где есть хотя бы один пропуск:")
rows_with_missing

Пропуски по столбцам:
order_id    0
client      0
city        1
category    0
product     0
quantity    1
price       1
manager     1
dtype: int64

Строки, где есть хотя бы один пропуск:


,order_id,client,city,category,product,quantity,price,manager
1,102,ООО Бета,NaN,Мониторы,Монитор 24,1.0,18000.0,Борис
2,103,ИП Гамма,Уфа,Периферия,Мышь Pro,NaN,1500.0,Анна
3,104,ООО Дельта,Самара,Ноутбуки,Notebook Pro,1.0,NaN,Виктор
7,107,ООО Тета,Пермь,Периферия,Кабель HDMI,5.0,900.0,NaN


## Практическая ячейка 3. Заполняем пропуски и показываем пример `dropna`

### Что делаем
Создаём копию таблицы и заполняем пропуски:
- `city` → `"Не указан"`
- `manager` → `"Не назначен"`
- `quantity` → `1`
- `price` → `0`

Также отдельно покажем пример `dropna(...)`: как удалить строки, где не хватает `quantity` или `price`.

### Функции, методы и синтаксис
- `df.copy()` — создаёт копию таблицы, чтобы не менять исходную.
- `df.fillna(value)` — заменяет пропуски на указанное значение.
- `df.dropna(subset=[...])` — удаляет строки, где есть пропуски в указанных столбцах.
- `df.isna().sum()` — позволяет проверить, остались ли пропуски после заполнения.


In [ ]:
df_filled = df.copy()

df_drop_example = df.dropna(subset=["quantity", "price"])

df_filled["city"] = df_filled["city"].fillna("Не указан")
df_filled["manager"] = df_filled["manager"].fillna("Не назначен")
df_filled["quantity"] = df_filled["quantity"].fillna(1)
df_filled["price"] = df_filled["price"].fillna(0)

print("Размер таблицы после dropna по столбцам quantity и price:", df_drop_example.shape)
print("\nОсталось пропусков после fillna:")
print(df_filled.isna().sum())

df_filled

Размер таблицы после dropna по столбцам quantity и price: (6, 8)

Осталось пропусков после fillna:
order_id    0
client      0
city        0
category    0
product     0
quantity    0
price       0
manager     0
dtype: int64


,order_id,client,city,category,product,quantity,price,manager
0,101,ООО Альфа,Москва,Ноутбуки,Notebook Air,2.0,55000.0,Анна
1,102,ООО Бета,Не указан,Мониторы,Монитор 24,1.0,18000.0,Борис
2,103,ИП Гамма,Уфа,Периферия,Мышь Pro,1.0,1500.0,Анна
3,104,ООО Дельта,Самара,Ноутбуки,Notebook Pro,1.0,0.0,Виктор
4,105,ИП Омега,Омск,Периферия,Клавиатура,4.0,3200.0,Анна
5,105,ИП Омега,Омск,Периферия,Клавиатура,4.0,3200.0,Анна
6,106,ООО Сигма,Казань,Мониторы,Монитор 27,3.0,21000.0,Елена
7,107,ООО Тета,Пермь,Периферия,Кабель HDMI,5.0,900.0,Не назначен


## Практическая ячейка 4. Находим и удаляем дубликаты

### Что делаем
Проверяем, есть ли повторяющиеся строки, выводим их отдельно и создаём новую таблицу без дубликатов.

### Функции, методы и синтаксис
- `df.duplicated()` — возвращает булеву маску: `True` для повторяющихся строк.
- `df[mask]` — показывает только те строки, где условие истинно.
- `df.drop_duplicates()` — удаляет повторяющиеся строки.
- `len(df)` — количество строк в таблице.


In [ ]:
duplicate_mask = df_filled.duplicated()
duplicate_rows = df_filled[duplicate_mask]

df_nodup = df_filled.drop_duplicates()

print("Количество дубликатов:", duplicate_mask.sum())
print("\nСтроки-дубликаты:")
print(duplicate_rows if not duplicate_rows.empty else "Дубликатов нет")
print("\nРазмер таблицы после удаления дубликатов:", df_nodup.shape)

df_nodup

Количество дубликатов: 1

Строки-дубликаты:
   order_id    client  city   category     product  quantity   price manager
5       105  ИП Омега  Омск  Периферия  Клавиатура       4.0  3200.0    Анна

Размер таблицы после удаления дубликатов: (7, 8)


,order_id,client,city,category,product,quantity,price,manager
0,101,ООО Альфа,Москва,Ноутбуки,Notebook Air,2.0,55000.0,Анна
1,102,ООО Бета,Не указан,Мониторы,Монитор 24,1.0,18000.0,Борис
2,103,ИП Гамма,Уфа,Периферия,Мышь Pro,1.0,1500.0,Анна
3,104,ООО Дельта,Самара,Ноутбуки,Notebook Pro,1.0,0.0,Виктор
4,105,ИП Омега,Омск,Периферия,Клавиатура,4.0,3200.0,Анна
6,106,ООО Сигма,Казань,Мониторы,Монитор 27,3.0,21000.0,Елена
7,107,ООО Тета,Пермь,Периферия,Кабель HDMI,5.0,900.0,Не назначен


## Практическая ячейка 5. Приводим типы данных и считаем выручку

### Что делаем
После заполнения пропусков и удаления дублей приводим столбцы `quantity` и `price` к типу `int`, а затем создаём новый столбец `revenue`.

### Функции, методы и синтаксис
- `df.copy()` — создаёт новую копию для финальной версии таблицы.
- `df["column"].astype(int)` — приводит столбец к целому типу.
- `df["new_column"] = ...` — создаёт новый столбец.
- `df["revenue"] = df["quantity"] * df["price"]` — поэлементный расчёт по столбцам.
- `df["revenue"].sum()` — сумма по столбцу.


In [ ]:
df_clean = df_nodup.copy()

df_clean["quantity"] = df_clean["quantity"].astype(int)
df_clean["price"] = df_clean["price"].astype(int)
df_clean["revenue"] = df_clean["quantity"] * df_clean["price"]

print("Типы данных после очистки:")
print(df_clean.dtypes)

print("\nИтоговая выручка:", df_clean["revenue"].sum())

df_clean

Типы данных после очистки:
order_id     int64
client      object
city        object
category    object
product     object
quantity     int64
price        int64
manager     object
revenue      int64
dtype: object

Итоговая выручка: 209800


,order_id,client,city,category,product,quantity,price,manager,revenue
0,101,ООО Альфа,Москва,Ноутбуки,Notebook Air,2,55000,Анна,110000
1,102,ООО Бета,Не указан,Мониторы,Монитор 24,1,18000,Борис,18000
2,103,ИП Гамма,Уфа,Периферия,Мышь Pro,1,1500,Анна,1500
3,104,ООО Дельта,Самара,Ноутбуки,Notebook Pro,1,0,Виктор,0
4,105,ИП Омега,Омск,Периферия,Клавиатура,4,3200,Анна,12800
6,106,ООО Сигма,Казань,Мониторы,Монитор 27,3,21000,Елена,63000
7,107,ООО Тета,Пермь,Периферия,Кабель HDMI,5,900,Не назначен,4500


## Ячейка 6. Тест и самопроверка

### Что делаем
Проверяем, что вы умеете:
- загружать Excel-лист;
- находить пропуски;
- заполнять пропуски;
- удалять дубликаты;
- приводить типы данных;
- считать итоговую выручку.

### Функции, методы и синтаксис
- `assert условие` — если условие ложное, Python покажет ошибку.
- `df.shape`, `df.isna().sum()`, `drop_duplicates()`, `astype()` — основные проверки по теме занятия.


In [ ]:
import pandas as pd

test_df = pd.read_excel("/content/drive/MyDrive/Базы/lesson_07_data_cleaning.xlsx", sheet_name="dirty_data", header=1)

assert test_df.shape == (8, 8)
assert test_df.isna().sum()["city"] == 1
assert test_df.isna().sum()["quantity"] == 1
assert test_df.isna().sum()["price"] == 1
assert test_df.isna().sum()["manager"] == 1

test_filled = test_df.copy()
test_filled["city"] = test_filled["city"].fillna("Не указан")
test_filled["manager"] = test_filled["manager"].fillna("Не назначен")
test_filled["quantity"] = test_filled["quantity"].fillna(1)
test_filled["price"] = test_filled["price"].fillna(0)

assert int(test_filled.isna().sum().sum()) == 0

test_nodup = test_filled.drop_duplicates().copy()
assert len(test_nodup) == 7

test_nodup["quantity"] = test_nodup["quantity"].astype(int)
test_nodup["price"] = test_nodup["price"].astype(int)
test_nodup["revenue"] = test_nodup["quantity"] * test_nodup["price"]

assert str(test_nodup["quantity"].dtype).startswith("int")
assert str(test_nodup["price"].dtype).startswith("int")
assert int(test_nodup["revenue"].sum()) == 209800

print("Тест пройден: данные очищены корректно, дубликаты удалены, выручка посчитана верно.")

Тест пройден: данные очищены корректно, дубликаты удалены, выручка посчитана верно.
